# RAG
Retrieval Augmented Generation (검색증강생성)

# import

In [1]:
import os
from dotenv import load_dotenv
print(load_dotenv())

from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.ai import AIMessage

from langchain_core.runnables.passthrough import RunnablePassthrough

True


In [16]:
# 1. Retrieval 단계
# private 으로부터 제공된 data 를 사용하거나 탐색함으로써
# language model 의 능력을 더 '확장(augment)'

# 2. Augmented Generation
# Model 로 하여금 '우리가 보낸 문서 data 만'을 가지고 답변하도록 할수도 있다.
# (경우에 따라, 우리 문서가 더 최신 data 일수도 있기 때문이다)
# 이를 통해 Model 이 과거에 학습한 data 를 참조하지 않게도 할수 있다.

In [17]:
"""
RAG

"What is 김정준?"

단순히 '김정준' 이란 단어를 갖는 문서를 찾는게 아니다

모델 은 '김정준' 이 뭔지 모른다
'김정준' 과 관련된 문서들을 retrieve 하여 context 로서 question 과 함께 묶어서 LLM 에 보냄

"""

'\nRAG\n\n"What is 김정준?"\n\n단순히 \'김정준\' 이란 단어를 갖는 문서를 찾는게 아니다\n\n모델 은 \'김정준\' 이 뭔지 모른다\n\'김정준\' 과 관련된 문서들을 retrieve 하여 context 로서 question 과 함께 묶어서 LLM 에 보냄\n\n'

In [18]:
# RAG 는 특정 라이브러리나 프레임워크 이름이 아니라
# 위와 같은 작업을 하는 '기법'을 일반적으로 통칭하는 용어

# RAG 를 수행하는 방법은 굉~장히 많고 다양.

In [19]:
# 어떤 방식으로 RAG 를 구현할른지는

# - 우리가 얼마나 많은 문서들을 가지고 있는지
# - 우리가 얼마나 많은 비용으로 운영할지 (어떤 모델, 가용한 token 개수등..)

# 등에 따라 결정될 문제다.


# Retrieve 란
![](https://miro.medium.com/v2/resize:fit:1100/format:webp/1*qyXS4oRtrW2NhhMRBxsdQQ.png)

In [20]:
# RAG 의 첫번째 단계인 Retrieval 의 일반적인 과정
# - data source 에서 데이터 load
# - 데이터는 split 하면서 transform
# - transform 한 데이터를 embed.
# - embed 된 데이터를 store 에 저장.
# - 검색(질의) 가 입력되면 store 에서 관련 문서들을 retrieve!

# Data Loaders

## 파일준비

In [21]:
# 아래와 같이 파일들을 준비합니다

# 출처는  조지오웰의 소설 '1984' Part1 Chapter1
#  http://www.george-orwell.org/1984/0.html

# 너무 길거나, 너무 짧지 않으면 좋습니다
# 파일이 너무 길면 나중에 임베딩 과정에서 비용지출이 발생.

# 다운로드 링크
# https://www.dropbox.com/scl/fi/ppid6hk7bwqxc0xrv65oc/files.zip?rlkey=df5d411n1fnaht2wlwv0o71wt&st=ddd52gw3&dl=1


In [22]:
llm = ChatOpenAI(temperature=0.1)

base_path = r'/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files'

## TextLoader

In [23]:
from langchain_community.document_loaders.text import TextLoader

In [24]:
# Data Loader 객체 생성
loader = TextLoader(os.path.join(base_path, 'chapter_one.txt'))

In [27]:
docs = loader.load()
docs
# ↓ List[Document] 객체  리턴

[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy dr

In [28]:
len(docs) # 1개의 Document

1

In [29]:
docs[0].page_content

"Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above

## PyPDFLoader

In [30]:
from langchain_community.document_loaders.pdf import PyPDFLoader

In [33]:
loader = PyPDFLoader(os.path.join(base_path, 'chapter_one.pdf'))
docs = loader.load()
docs

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2025-01-30T23:19:00+09:00', 'author': 'Yeonchul Sung', 'moddate': '2025-01-30T23:19:00+09:00', 'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Part 1, Chapter 1 \n \n \nPart One \n \n \n1 \nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his \nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through \nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of \ngritty dust from entering along with him. \n \nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured \nposter, too large for indoor display, had been tacked to the wall. It depicted simply an \nenormous face, more than a metre wide: the face of a man of about forty-five, with a \nheavy bl

## UnstructuredFileLoader

In [34]:
# 서로 다른 타입의 문서를 읽어오기 위해 각각의 DataLoader 를 사용하기 보다
# UnstructuredFileLoader 라는 것도 사용해볼수 있다. -> 꽤 다양한 포맷의 파일을 읽어올 수 있다

In [35]:
from langchain_community.document_loaders.unstructured import UnstructuredFileLoader

In [36]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.pdf'))
docs = loader.load()
docs

/var/folders/q9/xj42kkbn28g2gdfhyypn9yt80000gn/T/ipykernel_66370/3153963018.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.pdf'))
Matplotlib is building the font cache; this may take a moment.
No languages specified, defaulting to English.


[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.pdf'}, page_content="Part 1, Chapter 1\n\nPart One\n\n1\n\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his\n\nchin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through\n\nthe glass doors of Victory Mansions, though not quickly enough to prevent a swirl of\n\ngritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured\n\nposter, too large for indoor display, had been tacked to the wall. It depicted simply an\n\nenormous face, more than a metre wide: the face of a man of about forty-five, with a\n\nheavy black moustache and ruggedly handsome features. Winston made for the stairs. It\n\nwas no use trying the lift. Even at the best of times it was seldom working, and at\n\npresent the electric current was cut off during daylight hours. It wa

In [37]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.txt'))
docs = loader.load()
docs

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.txt'}, page_content="Part 1, Chapter 1\n\nPart One\n\n1 It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive

In [38]:
loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.docx'))
docs = loader.load()
docs

[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy d

# Splitter

## data를 split 해야 하는 이유

In [39]:
# 특정 질문에 답해야 하기 위해서, 필요한 '파일의 일부분' 만들 전달해야 할 수도 있다.

#  그래서 문서를 쪼개두어야(split) 한다

# 가령: "Ministry of peace" 를 찾고자 한다면.
# 해당 키워드가 있는 문서(들)만 모델에 넘겨주면 된다.

# 작은 조각들로 쪼개어 두면 필요한 것들을 찾기가 용이해진다.
#  - prompt 도 짧아질거다 (적은 token 사용, 적은 비용.)

# split 하는 방법은 다양하다.


## RecursiveCharaterTextSplitter

In [40]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [41]:
splitter = RecursiveCharacterTextSplitter()

# RecursiveCharacterTextSplitter 는 파일을 split 해주는데
# 문장의 끝이나, 문단의 끝부분마다 끊어준다.
# 문장 중간을 끊지는 않는다.  최대한 문장 중간에서 split 되지 않도록 하려 한다.
# 문장 중간에 짤림으로 의미있는 문장들을 잃고 싶지 않다.

# ↓ splitter 사용방법은 두가지 가 있다.

In [42]:
docs  # -> List[Document]

[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy d

In [43]:
len(docs)

1

### 방법 1 load(), split_documents()

In [44]:
documents = splitter.split_documents(docs)  # -> list[Document]
print(len(documents))
documents

11


[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy d

### 방법 2 load_and_split()

In [45]:
documents = loader.load_and_split(text_splitter=splitter)  # -> split된 List[Document]
print(len(documents))
documents

11


[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content="Part 1, Chapter 1\n\nPart One\n\n\n1\nIt was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.\n\nThe hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy d

In [47]:
print(documents[0].page_content)

Part 1, Chapter 1

Part One


1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.

The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his righ

### chunk_size=

In [49]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,  # 얼마나 큰 덩어리로 나눌 지 지정
                     # chunk_size의 단위는 splitter마다 다르다.
                     # CharacterTextSplitter계열의 경우 chunk_size는 '문자의 개수'
)

documents = loader.load_and_split(text_splitter=splitter)  # -> split된 List[Document]
print(len(documents))
documents

3498


[Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content='Part 1, Chapter 1\n\nPart One'),
 Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content='1'),
 Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content='It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors'),
 Document(metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content='was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of'),
 Document(metadata={'source': '/User

In [ ]:
# ↑ 문제점: 문단의 중간이 잘려버렸다 -> 문장의 의미가 파괴된다.

### chunk_overlap=

In [50]:
# 작은 덩어리이면서 문장의 중간을 잘라먹지 않는 방법은?
# chunk_overlap=
#    split 할때 앞 조각의 일부를 가져와서 연결해준다.
#    Document 간의 겹치는 부분 생길수 있다.

In [52]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
)

documents = loader.load_and_split(text_splitter=splitter)  # -> split된 List[Document]
print(len(documents))

for document in documents[10:50]:
    print('🟡', document.page_content)

250
🟡 move. BIG BROTHER IS WATCHING YOU, the caption beneath it ran.
🟡 Inside the flat a fruity voice was reading out a list of figures which had something to do with the production of pig-iron. The voice came from an oblong metal plaque like a dulled mirror which
🟡 an oblong metal plaque like a dulled mirror which formed part of the surface of the right-hand wall. Winston turned a switch and the voice sank somewhat, though the words were still distinguishable.
🟡 though the words were still distinguishable. The instrument (the telescreen, it was called) could be dimmed, but there was no way of shutting it off completely. He moved over to the window: a
🟡 it off completely. He moved over to the window: a smallish, frail figure, the meagreness of his body merely emphasized by the blue overalls which were the uniform of the party. His hair was very
🟡 were the uniform of the party. His hair was very fair, his face naturally sanguine, his skin roughened by coarse soap and blunt razor blades 

In [53]:
# ↑ Document 간에 겹치는 부분이 있다.
# 앞 Document 의 뒷부분을 가져다가 다음 Document 의 앞에 넣었다.
# 이렇게 하므로 문장의 (의미적) 구조를 크게 해치지 않도록 split 했다.

## CharacterTextSplitter

In [54]:
from langchain_text_splitters.character import CharacterTextSplitter

In [55]:
# CharacterTextSplitter 도 동작방식은 비슷하다
# separator=  : 특정 문자열 찾은 다음 이를 기준으로 분할한다.
splitter = CharacterTextSplitter(
    separator='\n',  # 줄 바꿈 단락별로 쪼갬
    chunk_size=600,
    chunk_overlap=100,
)

documents = loader.load_and_split(text_splitter=splitter)
print(len(documents))

for document in documents[0:5]:
    print('🟡', document.page_content)

Created a chunk of size 963, which is longer than the specified 600
Created a chunk of size 774, which is longer than the specified 600
Created a chunk of size 954, which is longer than the specified 600
Created a chunk of size 922, which is longer than the specified 600
Created a chunk of size 881, which is longer than the specified 600
Created a chunk of size 821, which is longer than the specified 600
Created a chunk of size 700, which is longer than the specified 600
Created a chunk of size 745, which is longer than the specified 600
Created a chunk of size 735, which is longer than the specified 600
Created a chunk of size 671, which is longer than the specified 600
Created a chunk of size 991, which is longer than the specified 600
Created a chunk of size 990, which is longer than the specified 600
Created a chunk of size 1289, which is longer than the specified 600
Created a chunk of size 1605, which is longer than the specified 600
Created a chunk of size 1900, which is longer 

46
🟡 Part 1, Chapter 1
Part One
1
It was a bright cold day in April, and the clocks were striking thirteen. Winston Smith, his chin nuzzled into his breast in an effort to escape the vile wind, slipped quickly through the glass doors of Victory Mansions, though not quickly enough to prevent a swirl of gritty dust from entering along with him.
🟡 The hallway smelt of boiled cabbage and old rag mats. At one end of it a coloured poster, too large for indoor display, had been tacked to the wall. It depicted simply an enormous face, more than a metre wide: the face of a man of about forty-five, with a heavy black moustache and ruggedly handsome features. Winston made for the stairs. It was no use trying the lift. Even at the best of times it was seldom working, and at present the electric current was cut off during daylight hours. It was part of the economy drive in preparation for Hate Week. The flat was seven flights up, and Winston, who was thirty-nine and had a varicose ulcer above his r

### length_function=

In [58]:
# splitter 에 length 를 계산하는 함수를 제공해줄수 있다.
#  length_function=   
#    기본적으론 파이썬의 len() 을 사용한다 (디폴트)


splitter = CharacterTextSplitter(
    separator='\n',
    chunk_size=600,
    chunk_overlap=100,
    length_function=len  # <- 기본적인 chunk 계수 함수
)

In [59]:
# 디폴트로 len() 함수가 동작함. CharacterTextSplitter 에선 '글자의 개수'를 chunk 카운트 함.
# 그러나 LLM 에서 말하는 token 은 문자(letter) 와는 다르다.
# 어떤 경우에는 문자 두개, 혹은 세개...  가 한개의 token 으로 카운트 된다.

## TikToken

### OpenAi Tokenizer 예시

In [60]:
# OpenAI 에서의 token 예시
# https://platform.openai.com/tokenizer
# model 의 관점에서, 몇개의 token 을 사용하는지 확인해 볼수 있다.

### from_tiktoken_encoder()

In [61]:
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator='\n',
    chunk_size=600,  # 600 token
    chunk_overlap=100,  # 100 token
)

In [62]:
# 이제 '모델'이 텍스트를 세는 방법과 'splitter'가 텍스트를 세는 방법이 일치 하게 되었다.
# model 에는 입력 limit 이 있기 때문에  (context window), 원하는 텍스트들을 모두 한번에 입력할 수는 없다.
# 그래서 우리 텍스트를 길이 계산할때 model 과 같은 방법으로 계산하는게 더 좋다.

# Embeddings

## Embedding과 Vector

In [ ]:
# Embedding 은 사람이 읽는 텍스트를 컴퓨터가 이해(연산)할 수 있는 숫자들(벡터)로 변환하는 작업이다.
# 우리가 만든 Document 마다 각각의 벡터를 만들어 주게 될겁니다.
# OpenAI의 Embedding 모델은 크기가 최소 1000차원 이상!의 벡터를 제공해준다.

In [ ]:
"""
3개의 차원을 정의해보자

첫번째 차원을 Masculinity (남성성)
두번째 차원을 Femininity (여성성)
세번째 차원을 Royalty (왕족스러움)

이제 특정 단어에 대한 차원 값(점수)를 줘보자

        Masculinity | Femininity  | Royalty
king  | 0.9         | 0.1         | 1.0
queen | 0.1         | 0.9         | 1.0
man   | 0.9         | 0.1         | 0.0
woman | 0.1         | 0.9         | 0.0

이렇게 3차원 벡터에 점수를 매겨 보았다.

단어를 이렇게 벡터로 점수를 매기면 '연산'을 할수 있게 된다.
king - man

               Masculinity | Femininity  | Royalty
king - man  =>  0.0        | 0.0         | 1.0  ===> 거의 'royal'

royal + woman => 0.1       | 0.9         | 1.0  ===> 거의 'queen'

★텍스트를 벡터화 하니까 '의미에 대한 연산'이 가능해진다!

"""
None

![](https://miro.medium.com/v2/resize:fit:2000/1*SYiW1MUZul1NvL1kc1RxwQ.png)


## word2vec 예시

https://turbomaze.github.io/word2vecjson/

## OpenAIEmbeddings

In [63]:
from langchain_openai.embeddings.base import OpenAIEmbeddings

In [64]:
embeddings = OpenAIEmbeddings()  # OPENAI_API_KEY 환경변수 있어야 함

In [65]:
embeddings.model

# text-embedding-ada-002'   
#  https://platform.openai.com/docs/models/text-embedding-ada-002
#  1M token 당 $0.1


'text-embedding-ada-002'

In [66]:
# OpenAIEmbeddings 를 통해
#  embed_documents()  <- 문서를 embed 하는것 뿐만 아니라
#  embed_query()      <- query 도 embed 하는 것이 가능하다.

In [68]:
vector = embeddings.embed_query('Hi')  # -> list[float]

print(vector)

[-0.0363546647131443, -0.007160174660384655, -0.03377465531229973, -0.02864069864153862, -0.02679038979113102, 0.03458253666758537, -0.0124635249376297, -0.007837752811610699, 0.0019187183352187276, -0.002667963271960616, 0.02471856400370598, -0.0024643640499562025, -0.005788730923086405, -0.0029920930974185467, 0.00666176388040185, -0.0030214113648980856, 0.03385283797979355, -0.0015408382751047611, 0.021057037636637688, -0.00903654471039772, -0.02173461578786373, 0.010365639813244343, 0.006283883936703205, 0.007081992458552122, -0.012261554598808289, 0.0008380140643566847, 0.005876685492694378, -0.009877001866698265, -0.003068646416068077, -0.02475765533745289, 0.01081518642604351, -0.013786105439066887, -0.024470988661050797, -0.014111863449215889, 0.002454591216519475, -0.018998242914676666, 0.0005786287947557867, -0.011336400173604488, 0.01813824102282524, -0.00996169913560152, 0.013147618621587753, -0.011329885572195053, -0.009147302247583866, -0.009701091796159744, -0.0264516007

In [70]:
print(len(vector))  # 1536차원

1536


In [71]:
# 입력 각 단어/문장 마다 1536차원의 벡터로 임베딩하여 리턴

In [73]:
vectors = embeddings.embed_documents([
    'hi',
    '김정준 아침에 뭐 먹었을까?',
    '최홍묵 점심에 뭐 먹을까?'
])  # -> list[list[float]]

In [74]:
print(len(vectors))  # 1536차원

3


In [75]:
# 모든 입력에 대해 동일한 크기의 차원으로 임베딩
for vector in vectors:
    print(len(vector))

1536
1536
1536


In [76]:
# 코드를 실행할때마다 '매번' 문서 embedding 을 반복해서 수행하는건 매우 비효율적이다
#  => 시간 소요 + 또한 비용 지출

# 대신! 그 embeded 된 결과들을 '저장'해 줄겁니다.
# LangChain 은 embedding 한것들을 캐싱하는 기능을 제공해준다

# '동일 Document'는 가급적 한번만 embedding 해주는게 좋다.


# Vector Store

In [77]:
# 일단 벡터를 만들고 나서, 그것들을 캐시해주고, vector store 에 넣어주면,
# 우리가 '검색'을 할수 있다.
# 그리하여, '관련있는 문서'들만 찾아낼수 있게 되는 거다

# 랭체인은 다양한 vector store 를 제공한다,  어떤거는 cloud 형태이고, 어떤건 유료이기도 하다.

# 우리는 예제에서 무료로 사용할수 있고 로컬로 저장되는 Chroma 라는 것을 사용해볼겁니다
# 나중에 FAISS 라는 메모리기반 vector store 도 사용해볼거구..
# 클라우드기반의 vector store 인 pinecone 도 사용해 보자.

## Chroma vector store

In [79]:
from langchain_chroma import Chroma

In [78]:
# ↓이 ChromaDB 에 'split 된 문서' 와 'OpenAI embedding model' 을 전달해야 한다

# Chroma 에 전달해야 하는 것
#  1. 'split 된 Document들'
#  2. Embedding model

# OpenAIEmbeddings 의 옵션에 model= 이 있다. 여기에 원하는 모델 지정가능 (지정안하면 default 동작)

# ⭐️ embedding 모델을 사용하는것도 비용이 발생한다!

In [80]:
# 아래 코드 복사
splitter = CharacterTextSplitter.from_tiktoken_encoder(
    separator="\n",
    chunk_size=600,
    chunk_overlap=100,
)

loader = UnstructuredFileLoader(os.path.join(base_path, 'chapter_one.docx'))

# split 된 Document 준비
docs = loader.load_and_split(text_splitter=splitter)

# embedding 모델 준비
embeddings = OpenAIEmbeddings()


In [81]:
vectorstore = Chroma.from_documents(docs, embeddings)
# ↑ ⭐️ 이 코드 실행하면 비용지출 발생함.
#       문서의 크기가 클수록 당연히 비례해서 비용 발생
#       우리 예제에서는 작은 파일을 사용하는 것이니 매우 적은 비용이 발생할 것이다.

In [84]:
# 이제 vectorstore 를 사용하여 '유사도' 검색을 수행해보자
# 우리 Document 들이 벡터와 되었고, 이제 벡터공간에 대한 검색을 해볼수 있다.

# similarity_search 는 기본적으로 4개의 Document 리턴 (k=4)

results = vectorstore.similarity_search("where does winston live")  # -> list[Document]
print(len(results))
print(results)

4
[Document(id='cbc93130-d437-4505-be3d-4289fd46d9dd', metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content="The Ministry of Love was the really frightening one. There were no windows in it at all. Winston had never been inside the Ministry of Love, nor within half a kilometre of it. It was a place impossible to enter except on official business, and then only by penetrating through a maze of barbed-wire entanglements, steel doors, and hidden machine-gun nests. Even the streets leading up to its outer barriers were roamed by gorilla-faced guards in black uniforms, armed with jointed truncheons.\nWinston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the k

In [ ]:
# 위 검색된 Document 들이  LLM 에 전달될것이고  Retrieval!
# LLM 은 이 Document 기반의 답변을 할수 있게 될것이다 ! --> RAG

## embedding cache

In [85]:
# 다시 실행하면 임베딩 결과는 다 사라진다. 재실행하면 다시 재계산 발생 (비용발생!)
# 그래서 embedding 을 캐싱해주자

In [86]:
from langchain_classic.embeddings import CacheBackedEmbeddings

In [87]:
from langchain_classic.storage import LocalFileStore  # 파일로 cache

In [88]:
base_path

'/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files'

In [90]:
# 캐시 경로 지정. 여기에 embedding 결과가 캐시 될거다.
cache_dir = LocalFileStore(os.path.join(base_path, '.cache'))

cached_embeddings = CacheBackedEmbeddings.from_bytes_store(
    embeddings,  # 임베딩 모델
    cache_dir,  # 캐시 저장소
)

# Chroma 에는 아까는  split 된 문서 와 '임베딩 모델'을 건네주었지만
# 이번에는 위에서 만든 'cached embedding' 을 전달해주면 된다
vectorstore = Chroma.from_documents(docs, cached_embeddings)

# ↑ 이렇게 하면,  
#  최초에 Chroma.from_documents() 를 호출할때는 
#     OpenAIEmbeddings 을 사용하여 임베딩. 결과는 cache 함
#  다음에 Chroma.from_documents() 를 호출할때는
#      OpenAIEmbeddings 대신에 미리 cache 되어 있듣 embeddings 를 전달할거다.

# 위코드를 실행하여 우리가 또 파일 embedding 작업을 할때는,
# 1.첫번째로!
#   캐시에 embeddings 가 이미 존재하는지 확인할거다.
# 2.만약 없다면!
#    vector store(Chroma.from_documents) 를 호출할 때
#   문서들(docs) 과 함께 OpenAIEmbeddings 를 사용할거다.
#

results = vectorstore.similarity_search("where does winston live")
results


[Document(id='e49caa2f-6096-4efb-8c3e-a25cef04251e', metadata={'source': '/Users/leo/LeoData/MyCode/KoreaITAcademy/LangChain/dataset/files/chapter_one.docx'}, page_content="The Ministry of Love was the really frightening one. There were no windows in it at all. Winston had never been inside the Ministry of Love, nor within half a kilometre of it. It was a place impossible to enter except on official business, and then only by penetrating through a maze of barbed-wire entanglements, steel doors, and hidden machine-gun nests. Even the streets leading up to its outer barriers were roamed by gorilla-faced guards in black uniforms, armed with jointed truncheons.\nWinston turned round abruptly. He had set his features into the expression of quiet optimism which it was advisable to wear when facing the telescreen. He crossed the room into the tiny kitchen. By leaving the Ministry at this time of day he had sacrificed his lunch in the canteen, and he was aware that there was no food in the kit

In [91]:
import glob
for cached_file in glob.glob(os.path.join(base_path, '.cache', '*')):
  with open(cached_file, 'r') as f:
    print(f.read())
    break

[-0.02668767422437668, 0.0021690530702471733, 0.009077070280909538, -0.014172731898725033, -0.02975866012275219, 0.025260889902710915, -0.008479179814457893, -0.025315243750810623, -0.020097287371754646, -0.012018965557217598, 0.004382268525660038, 0.017773665487766266, -0.0029028281569480896, -0.0034412697423249483, 0.004317723214626312, 0.018901504576206207, 0.0460919514298439, -0.008492767810821533, 0.025682130828499794, -0.01451244205236435, 0.009049894288182259, 0.009946729987859726, -0.012684798799455166, 0.02187737077474594, -0.008119086734950542, -0.014784211292862892, 0.008309324271976948, -0.02069517783820629, 0.011169688776135445, -0.007602726109325886, -0.0022675690706819296, -0.009640990756452084, 0.0013537472113966942, -0.0027856279630213976, -0.040928348898887634, -0.010619357228279114, -0.0001469248963985592, -0.028426993638277054, 0.006237089168280363, -0.023412862792611122, 0.027720395475625992, -0.002021278953179717, -0.003529594512656331, -0.011896669864654541, -0.0